# Transform payment data
1. Extract date and time from payment_timestamp and create new column payment_date and payment_time
2. Map payment_status to contain description values
(1-success, 2-pending, 3-cancelled, 4-failed)
3. Write transformed data to the silver schema

In [0]:
%sql
SELECT
payment_id,
order_id,
payment_timestamp,
payment_status,
payment_method
FROM gizmobox.bronze.payments
LIMIT 5;


payment_id,order_id,payment_timestamp,payment_status,payment_method
35,13,2024-12-17T05:25:34Z,2,PayPal
36,16,2024-12-09T10:45:56Z,4,Bank Transfer
37,17,2024-12-02T13:50:20Z,1,Bank Transfer
38,24,2024-12-22T08:30:55Z,1,PayPal
39,27,2024-12-03T11:45:30Z,2,Credit Card


#### 1. Extract date and time from payment_timestamp and create new column payment_date and payment_time

In [0]:
%sql
SELECT
payment_id,
order_id,
CAST(date_format(payment_timestamp, 'yyyy-MM-dd') AS DATE) payment_data,
date_format(payment_timestamp, 'HH-mm-ss') payment_time,
payment_status,
payment_method
FROM gizmobox.bronze.payments ;

payment_id,order_id,payment_data,payment_time,payment_status,payment_method
35,13,2024-12-17,05-25-34,2,PayPal
36,16,2024-12-09,10-45-56,4,Bank Transfer
37,17,2024-12-02,13-50-20,1,Bank Transfer
38,24,2024-12-22,08-30-55,1,PayPal
39,27,2024-12-03,11-45-30,2,Credit Card
40,28,2024-12-27,05-40-10,4,Credit Card
41,32,2024-12-29,09-20-40,1,PayPal
42,37,2024-12-23,12-10-05,2,Credit Card
43,38,2024-12-26,20-15-15,1,Credit Card
44,44,2024-12-21,14-25-45,1,Credit Card


#### 2. Map payment_status to contain description values
(1-success, 2-pending, 3-cancelled, 4-failed)

In [0]:
%sql
SELECT
payment_id,
order_id,
CAST(date_format(payment_timestamp, 'yyyy-MM-dd') AS DATE) payment_data,
date_format(payment_timestamp, 'HH-mm-ss') payment_time,
payment_status,
CASE payment_status
    WHEN 1 THEN 'Success'
    WHEN 2 THEN 'Pending'
    WHEN 3 THEN 'Cancelled'
    WHEN 4 THEN 'Failed' END AS payment_status_group,
payment_method
FROM gizmobox.bronze.payments ;

payment_id,order_id,payment_data,payment_time,payment_status,payment_status_group,payment_method
35,13,2024-12-17,05-25-34,2,Pending,PayPal
36,16,2024-12-09,10-45-56,4,Failed,Bank Transfer
37,17,2024-12-02,13-50-20,1,Success,Bank Transfer
38,24,2024-12-22,08-30-55,1,Success,PayPal
39,27,2024-12-03,11-45-30,2,Pending,Credit Card
40,28,2024-12-27,05-40-10,4,Failed,Credit Card
41,32,2024-12-29,09-20-40,1,Success,PayPal
42,37,2024-12-23,12-10-05,2,Pending,Credit Card
43,38,2024-12-26,20-15-15,1,Success,Credit Card
44,44,2024-12-21,14-25-45,1,Success,Credit Card


#### 3. Write transformed data to the silver schema

In [0]:
%sql
CREATE TABLE gizmobox.silver.payments
AS
SELECT
    payment_id,
    order_id,
    CAST(date_format(payment_timestamp, 'yyyy-MM-dd') AS DATE) payment_data,
    date_format(payment_timestamp, 'HH-mm-ss') payment_time,
    payment_status,
    CASE payment_status
        WHEN 1 THEN 'Success'
        WHEN 2 THEN 'Pending'
        WHEN 3 THEN 'Cancelled'
        WHEN 4 THEN 'Failed' END AS payment_status_group,
    payment_method
    FROM gizmobox.bronze.payments ;



num_affected_rows,num_inserted_rows
